# 🎓 EX50: การปรับแต่งไฮเปอร์พารามิเตอร์สำหรับ YOLO (Hyperparameter Tuning for YOLO)

ยินดีต้อนรับนักเรียนทุกคนเข้าสู่บทเรียน! วันนี้เราจะมาเจาะลึกในหัวข้อที่น่าสนใจเกี่ยวกับ **การปรับแต่งไฮเปอร์พารามิเตอร์ (Hyperparameter Tuning)** สำหรับแบบจำลองการตรวจจับวัตถุ YOLO

ต่างจากพารามิเตอร์ของแบบจำลอง (น้ำหนักและค่าไบแอส - weights and biases) ซึ่งเรียนรู้โดยอัตโนมัติผ่านการแพร่กระจายย้อนกลับ (backpropagation) ระหว่างการฝึกฝน ส่วน **ไฮเปอร์พารามิเตอร์ (hyperparameters)** คือการตั้งค่าทางสถาปัตยกรรมและการปรับค่าความเหมาะสมของแบบจำลอง (optimization settings) ที่เราต้องกำหนดค่าล่วงหน้าก่อนการฝึกฝน การค้นหาส่วนผสมที่เหมาะสมเป็นสิ่งสำคัญอย่างยิ่งในการเพิ่มประสิทธิภาพสูงสุดของแบบจำลอง โดยเฉพาะอย่างยิ่งเมื่อฝึกฝนกับชุดข้อมูลเฉพาะทาง

---

## 🔍 ไฮเปอร์พารามิเตอร์หลักของ YOLO

ในการปรับแต่งแบบจำลอง YOLO อย่างมีประสิทธิภาพ เราต้องเข้าใจกลไกต่าง ๆ ที่เราสามารถควบคุมได้:
1. **`lr0` (อัตราการเรียนรู้เริ่มต้น - Initial Learning Rate):** กำหนดขนาดก้าวในช่วงเริ่มต้นการฝึกฝน หากสูงเกินไป ค่าน้ำหนักจะผันผวนหรือแตกกระจาย (oscillate or explode) หากต่ำเกินไป การฝึกฝนจะช้ามากหรือติดอยู่ในจุดต่ำสุดท้องถิ่น (local minima)
2. **`lrf` (เศษส่วนอัตราการเรียนรู้สุดท้าย - Final Learning Rate Fraction):** อัตราการเรียนรู้สุดท้ายจะคำนวณจาก `lr0 * lrf` ซึ่งเป็นตัวกำหนดว่าอัตราการเรียนรู้จะลดลงเหลือเท่าใดเมื่อสิ้นสุดการฝึกฝน
3. **`momentum` (โมเมนตัมของการไล่ระดับสี - Gradient Momentum):** ตัวเร่งความเร็วสำหรับการอัปเดตค่าน้ำหนักเพื่อรักษาความเสถียรของการไล่ระดับสี (โดยทั่วไปตั้งค่าไว้ที่ `0.937`)
4. **`weight_decay` (การควบคุมน้ำหนัก - L2 Regularization):** ค่าปรับเพื่อป้องกันไม่ให้น้ำหนักมีขนาดใหญ่เกินไปเพื่อช่วยลดการฟิตเกิน (overfitting)
5. **`mosaic` (การขยายข้อมูลแบบโมเสก - Mosaic Augmentation):** ผสมผสานรูปภาพฝึกฝน 4 รูปเข้าเป็นรูปเดียว บังคับให้ YOLO เรียนรู้รายละเอียดของวัตถุที่ระดับขนาดต่าง ๆ และลดการพึ่งพาบริบทโดยรวมของรูปภาพ

## 📉 คณิตศาสตร์ของตัวกำหนดเวลาแบบ Cosine Annealing (Cosine Annealing Scheduler Math)

YOLO จะลดทอนอัตราการเรียนรู้ $\eta_t$ ที่รอบ (epoch) $t$ โดยใช้กำหนดการแบบ **Cosine Annealing**:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{T_{cur}}{T_{\max}}\pi\right)\right)$$

โดยที่:
* $\eta_{\max}$ คืออัตราการเรียนรู้เริ่มต้น (`lr0`)
* $\eta_{\min}$ คืออัตราการเรียนรู้สุดท้ายเป้าหมาย (`lr0 * lrf`)
* $T_{cur}$ คือดัชนีรอบปัจจุบัน (เริ่มจาก 0)
* $T_{\max}$ คือจำนวนรอบทั้งหมด

### ทำไมต้องเป็น Cosine Annealing?
ในช่วงเริ่มต้นการฝึกฝน เราต้องการอัตราการเรียนรู้ที่สูงเพื่อค้นหาแอ่งพื้นที่ที่ดีใน loss landscape อย่างรวดเร็ว เมื่อการฝึกฝนคืบหน้าไป เราต้องการลดอัตราการเรียนรู้ลงอย่างราบรื่นเพื่อปรับค่าพารามิเตอร์ได้อย่างละเอียดถี่ถ้วนยิ่งขึ้น เส้นโค้งโคไซน์สอดคล้องกับแนวคิดนี้เป็นอย่างดี: เริ่มต้นด้วยการลดทอนลงอย่างช้า ๆ เร่งความเร็วขึ้นตรงกลาง และแบนราบลงในช่วงท้าย

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def get_cosine_lr(epoch: int, total_epochs: int, lr0: float, lrf: float) -> float:
    t_cur = min(epoch, total_epochs)
    eta_max = lr0
    eta_min = lr0 * lrf
    cos_factor = math.cos((t_cur / total_epochs) * math.pi)
    return eta_min + 0.5 * (eta_max - eta_min) * (1.0 + cos_factor)

# Parameters
lr0 = 0.01
lrf = 0.01
total_epochs = 100

epochs = np.arange(total_epochs + 1)
lrs = [get_cosine_lr(e, total_epochs, lr0, lrf) for e in epochs]

plt.figure(figsize=(10, 5))
plt.plot(epochs, lrs, label='Cosine Annealing', color='teal', linewidth=2)
plt.axhline(y=lr0, color='r', linestyle='--', label=f'Initial LR (lr0 = {lr0})')
plt.axhline(y=lr0 * lrf, color='g', linestyle='--', label=f'Final LR (lr0*lrf = {lr0*lrf})')
plt.title('Cosine Annealing Learning Rate Decay Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

## 🧬 อัลกอริทึมพันธุกรรมสำหรับการค้นหาไฮเปอร์พารามิเตอร์ (Genetic Algorithms for Hyperparameter Search)

**อัลกอริทึมพันธุกรรม (Genetic Algorithm - GA)** เลียนแบบกระบวนการคัดเลือกตามธรรมชาติเพื่อค้นหาไฮเปอร์พารามิเตอร์ที่เหมาะสมที่สุดโดยอัตโนมัติ:
1. **การเริ่มต้น (Initialize):** เริ่มต้นด้วยชุดไฮเปอร์พารามิเตอร์พื้นฐาน (เรียกว่า "รุ่นพ่อแม่" หรือ "parent")
2. **การกลายพันธุ์ (Mutate):** สร้างรุ่นลูก (offspring) โดยการสุ่มเพิ่มสัญญาณรบกวน (noise) เข้าไปในพารามิเตอร์ของรุ่นพ่อแม่ เราสุ่มตัวอย่างจากการแจกแจงแบบปกติ (Normal distribution):
   $$\Delta \sim N(0, \sigma \cdot \text{range})$$
   โดยที่ $\text{range} = \text{upper\_bound} - \text{lower\_bound}$ และ $\sigma$ คือความแรงของการกลายพันธุ์ (mutation strength)
3. **การขลิบค่าขอบเขต (Clip):** ตรวจสอบให้แน่ใจว่าค่าที่กลายพันธุ์แล้วยังคงอยู่ในช่วงขอบเขตที่มีความหมายในทางกายภาพ (เช่น โมเมนตัมต้องอยู่ระหว่าง 0.6 ถึง 0.999; weight_decay >= 0.0)
4. **การประเมินผล (Evaluate):** ฝึกฝนแบบจำลองด้วยพารามิเตอร์ที่ผ่านการกลายพันธุ์เป็นเวลาสองสามรอบ และบันทึกค่าความเหมาะสม (fitness score) เช่น การนำค่า mAP@0.5 และ mAP@0.5:0.95 มาร่วมคำนวณ
5. **การคัดเลือก (Select):** รุ่นลูกที่มีประสิทธิภาพดีที่สุดจะกลายเป็นรุ่นพ่อแม่สำหรับเจเนอเรชันถัดไป

In [ ]:
import random
from typing import Dict, Tuple

def mutate_hyperparameters(
    parent_hyp: Dict[str, float],
    bounds: Dict[str, Tuple[float, float]],
    mutation_rate: float = 0.8,
    sigma: float = 0.1
) -> Dict[str, float]:
    mutated_hyp = {}
    for key, parent_val in parent_hyp.items():
        if key not in bounds:
            mutated_hyp[key] = parent_val
            continue
        lower, upper = bounds[key]
        param_range = upper - lower
        if random.random() < mutation_rate:
            mutation = random.gauss(0, sigma * param_range)
            mutated_val = parent_val + mutation
            clipped_val = max(lower, min(upper, mutated_val))
            mutated_hyp[key] = clipped_val
            assert lower <= clipped_val <= upper, "Out of bounds!"
        else:
            mutated_hyp[key] = parent_val
    return mutated_hyp

# Base/parent hyperparameters
parent_hyp = {'momentum': 0.937}
bounds = {'momentum': (0.6, 0.999)}

# Generate 10,000 mutations to visualize distribution
random.seed(42)
mutations = [mutate_hyperparameters(parent_hyp, bounds, mutation_rate=1.0, sigma=0.15)['momentum'] for _ in range(10000)]

plt.figure(figsize=(10, 5))
plt.hist(mutations, bins=50, color='darkslateblue', edgecolor='white', alpha=0.8)
plt.axvline(x=parent_hyp['momentum'], color='r', linestyle='--', label=f"Parent Momentum ({parent_hyp['momentum']})")
plt.axvline(x=bounds['momentum'][0], color='orange', linestyle=':', label='Lower Bound (0.6)')
plt.axvline(x=bounds['momentum'][1], color='orange', linestyle=':', label='Upper Bound (0.999)')
plt.title('Distribution of Mutated Momentum Hyperparameter (with Clipping)', fontsize=14)
plt.xlabel('Mutated Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 🔬 กรณีศึกษา: การปรับแต่ง YOLO สำหรับวัตถุทางอุตสาหกรรมของ PTT

มานำความรู้ของเราไปใช้กับตัวอย่างการทำงานจริง: การฝึกฝน YOLO บนชุดข้อมูล `overall-ptt-object-detection.v11i.yolov11` ซึ่งประกอบด้วยวัตถุที่มีขนาดเล็กและบาง เช่น `small-valve` (วาล์วขนาดเล็ก), `lever-handle` (ด้ามคันโยก) และ `pig-alert` (สัญญาณเตือนของอุปกรณ์ pig)

### คำแนะนำของอาจารย์ผู้สอน:
1. **ลดการขยายภาพเชิงพื้นที่โดยการสเกลรูปภาพ (`scale`):** โดยค่าเริ่มต้น YOLO จะสุ่มสเกลรูปภาพขึ้นและลง (เช่น `scale=0.5` จะอนุญาตให้ลดขนาดรูปภาพลงเหลือครึ่งหนึ่ง) สำหรับวาล์วหรือด้ามคันโยกขนาดจิ๋ว การลดขนาดพิกเซลลงจะทำให้พวกมันหายไปจากแผนที่ลักษณะเด่น (feature maps) โดยสิ้นเชิง เราควรกลายพันธุ์ค่า `scale` ให้เล็กลง (เช่น ขอบเขต `(0.1, 0.3)` แทนที่จะเป็น `(0.0, 0.9)`) เพื่อรักษาคุณลักษณะเชิงพื้นที่ของวัตถุจิ๋วให้สมบูรณ์
2. **ปรับแต่งจำนวนรอบในการปิดระบบโมเสก (`close_mosaic`):** การขยายข้อมูลแบบโมเสก (Mosaic augmentation) ผสมผสานรูปภาพฝึกฝน 4 รูปเข้าเป็นรูปเดียว แม้ว่าสิ่งนี้จะดีมากในการสอนแบบจำลองให้เรียนรู้ลักษณะเด่นโดยไม่ต้องพึ่งพาบริบทในช่วงเริ่มต้น แต่เส้นขอบของรูปภาพที่รวมเข้าด้วยกันอาจทำให้เกิดความสับสนต่อฟังก์ชันการสูญเสียของการระบุตำแหน่ง (localization loss) ในช่วงท้ายของการฝึกฝน การตั้งค่า `close_mosaic` (เช่น ปิดการใช้งาน mosaic ล่วงหน้า 15-20 รอบก่อนสิ้นสุดการฝึกฝน) จะช่วยให้แบบจำลองฝึกฝนอย่างละเอียด (fine-tune) บนขอบเขตวัตถุจริงที่สะอาดและไม่มีการบิดเบือน ซึ่งจะเพิ่มความแม่นยำของการระบุตำแหน่งสุดท้าย (mAP@0.5:0.95) ได้อย่างมีนัยสำคัญ

---
*หัวข้อที่เกี่ยวข้อง:*
* [[EX49_YOLO_Result_Analysis]] - การวิเคราะห์ผลลัพธ์การฝึกฝนและเส้นโค้งความสูญเสีย
* [[EX27_Learning_Rate]] - ความเข้าใจเกี่ยวกับหลักการของอัตราการเรียนรู้
* กลับสู่แผนการเรียนรู้หลัก: [[YOLO_Learning_Plan]]
* บันทึกความก้าวหน้า: [[learning_journal]]